In [1]:
# Upload the zip file
from google.colab import files
uploaded = files.upload()  # Select frames_cleaned.zip from your computer

# Extract the zip file
import zipfile
import os

zip_path = "frames_cleaned.zip"
extract_path = "/content/frames_cleaned"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Frames extracted successfully!")

# Verify the structure
!ls /content/frames_cleaned
!ls /content/frames_cleaned/OBJ
!ls /content/frames_cleaned/NON_OBJ

Saving frames_cleaned.zip to frames_cleaned.zip
✅ Frames extracted successfully!
frames
ls: cannot access '/content/frames_cleaned/OBJ': No such file or directory
ls: cannot access '/content/frames_cleaned/NON_OBJ': No such file or directory


In [2]:
import os
import pandas as pd

# Define the paths
obj_folder = "/content/frames_cleaned/frames/OBJ"
non_obj_folder = "/content/frames_cleaned/frames/NON_OBJ"

# Function to collect frame paths and labels
def collect_frame_data(folder, label):
    frame_data = []
    for video in os.listdir(folder):
        video_path = os.path.join(folder, video)
        if os.path.isdir(video_path):
            for frame in os.listdir(video_path):
                if frame.endswith('.jpg') or frame.endswith('.png'):
                    frame_path = os.path.join(video_path, frame)
                    frame_data.append({
                        'frame_path': frame_path,
                        'label': label
                    })
    return frame_data

# Collect data for OBJ (1) and NON_OBJ (0)
obj_data = collect_frame_data(obj_folder, 1)
non_obj_data = collect_frame_data(non_obj_folder, 0)

# Combine and create DataFrame
all_data = obj_data + non_obj_data
df = pd.DataFrame(all_data)

# Save to CSV
df.to_csv('frames_metadata.csv', index=False)
print("✅ frames_metadata.csv created successfully!")

# Preview the first few rows
print(df.head())

✅ frames_metadata.csv created successfully!
                                          frame_path  label
0  /content/frames_cleaned/frames/OBJ/video30/fra...      1
1  /content/frames_cleaned/frames/OBJ/video30/fra...      1
2  /content/frames_cleaned/frames/OBJ/video30/fra...      1
3  /content/frames_cleaned/frames/OBJ/video30/fra...      1
4  /content/frames_cleaned/frames/OBJ/video30/fra...      1


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import pandas as pd

# Assuming df is your original DataFrame loaded from 'frames_metadata.csv'
# Split into train and test (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Add a new column 'label_str' with string labels
train_df['label_str'] = train_df['label'].map({0: 'NON_OBJ', 1: 'OBJ'})
test_df['label_str'] = test_df['label'].map({0: 'NON_OBJ', 1: 'OBJ'})

# Define image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Test generator (no augmentation, just rescaling)
test_datagen = ImageDataGenerator(rescale=1./255)

# Training generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='frame_path',
    y_col='label_str',  # Use the new string label column
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

# Validation generator
validation_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='frame_path',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# Test generator
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='frame_path',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("✅ Data generators created!")

Found 2948 validated image filenames belonging to 2 classes.
Found 736 validated image filenames belonging to 2 classes.
Found 922 validated image filenames belonging to 2 classes.
✅ Data generators created!


In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Load MobileNetV2 pre-trained on ImageNet
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
base_model.trainable = False

# Add custom layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Show model summary
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Conv1 (Conv2D)            │ (None, 112, 112, 32)   │            864 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bn_Conv1                  │ (None, 112, 112, 32)   │            128 │ Conv1[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Conv1_relu (ReLU)         │ (None, 112, 112, 32)   │              0 │ bn_Conv1[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ expanded_conv_depthwise   │ (None, 112, 112, 32)   │            288 │ Conv1_relu[0][0]       │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ expanded_conv_depthwise_… │ (None, 112, 112, 32)   │            128 │ expanded_conv_depthwi… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ expanded_conv_depthwise_… │ (None, 112, 112, 32)   │              0 │ expanded_conv_depthwi… │
│ (ReLU)                    │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ expanded_conv_project     │ (None, 112, 112, 16)   │            512 │ expanded_conv_depthwi… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ expanded_conv_project_BN  │ (None, 112, 112, 16)   │             64 │ expanded_conv_project… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_expand (Conv2D)   │ (None, 112, 112, 96)   │          1,536 │ expanded_conv_project… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_expand_BN         │ (None, 112, 112, 96)   │            384 │ block_1_expand[0][0]   │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_expand_relu       │ (None, 112, 112, 96)   │              0 │ block_1_expand_BN[0][… │
│ (ReLU)                    │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_pad               │ (None, 113, 113, 96)   │              0 │ block_1_expand_relu[0… │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_depthwise         │ (None, 56, 56, 96)     │            864 │ block_1_pad[0][0]      │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block_1_depthwise_BN 

 Total params: 2,586,177 (9.87 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# Set epochs
EPOCHS = 20

# Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

print("✅ Training completed!")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 73s 668ms/step - accuracy: 0.6149 - loss: 0.7774 - val_accuracy: 0.7065 - val_loss: 0.5766
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 492ms/step - accuracy: 0.7029 - loss: 0.5680 - val_accuracy: 0.7133 - val_loss: 0.5467
Epoch 3/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 44s 477ms/step - accuracy: 0.7405 - loss: 0.5069 - val_accuracy: 0.7201 - val_loss: 0.5126
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 490ms/step - accuracy: 0.7554 - loss: 0.5111 - val_accuracy: 0.7541 - val_loss: 0.5086
Epoch 5/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 491ms/step - accuracy: 0.7629 - loss: 0.4750 - val_accuracy: 0.7486 - val_loss: 0.4748
Epoch 6/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 45s 481ms/step - accuracy: 0.7805 - loss: 0.4525 - val_accuracy: 0.7717 - val_loss: 0.4684
Epoch 7/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 45s 479ms/step - accuracy: 0.7870 - loss: 0.4383 - val_accuracy: 0.7772 - val_loss: 0.4512
Epoch 8/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 497ms/step - accuracy: 0.8135 - loss: 0.4002 - val_accu

In [7]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

29/29 ━━━━━━━━━━━━━━━━━━━━ 6s 211ms/step - accuracy: 0.8468 - loss: 0.3476
Test Accuracy: 83.95%


In [8]:
# Unfreeze the last 20 layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile with a lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

# Fine-tune
history_fine = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

print("✅ Fine-tuning completed!")

Epoch 1/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 69s 591ms/step - accuracy: 0.7238 - loss: 0.5783 - val_accuracy: 0.8370 - val_loss: 0.3774
Epoch 2/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 67s 499ms/step - accuracy: 0.7933 - loss: 0.4497 - val_accuracy: 0.8220 - val_loss: 0.3964
Epoch 3/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 82s 498ms/step - accuracy: 0.8233 - loss: 0.3747 - val_accuracy: 0.8288 - val_loss: 0.4042
Epoch 4/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 47s 504ms/step - accuracy: 0.8195 - loss: 0.3800 - val_accuracy: 0.8370 - val_loss: 0.3585
Epoch 5/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 47s 510ms/step - accuracy: 0.8369 - loss: 0.3839 - val_accuracy: 0.8533 - val_loss: 0.3774
Epoch 6/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 491ms/step - accuracy: 0.8513 - loss: 0.3277 - val_accuracy: 0.8329 - val_loss: 0.3862
Epoch 7/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 82s 490ms/step - accuracy: 0.8615 - loss: 0.3241 - val_accuracy: 0.8410 - val_loss: 0.3622
Epoch 8/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 47s 506ms/step - accuracy: 0.8582 - loss: 0.3283 - val_accu

In [9]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 128ms/step - accuracy: 0.8549 - loss: 0.3522
Test Accuracy: 85.68%


In [10]:
model.save('objectification_model.h5')
print("✅ Model saved successfully!")

✅ Model saved successfully!


In [21]:
import numpy as np

# Load the model
loaded_model = tf.keras.models.load_model('objectification_model.h5')

# Prediction function
def predict_frame(frame_path):
    img = tf.keras.preprocessing.image.load_img(frame_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = loaded_model.predict(img_array)
    return 'OBJ' if prediction[0][0] < 0.5 else 'NON_OBJ'

# Test it
frame_path = '/content/frames_cleaned/frames/OBJ/video31/frame_15.jpg'  # Adjust path as needed
result = predict_frame(frame_path)
print(f"Prediction for {frame_path}: {result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Prediction for /content/frames_cleaned/frames/OBJ/video31/frame_15.jpg: OBJ
